In [4]:
# PyTorch functions/methods helpers

# 8.1.2
# conv2d(24, 24...) and linear(32, 32) are not redundant in these cases because they allowed the model more opportunities to learn sophisticated features before reducing the outputs (to 16 and 10)

alexnet_small = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=11, stride=4, padding=2), nn.ReLU(), # output_size = floor((96 + 2*2 - 11) / 4 + 1) = floor(23.25) = 23 for shape of (batch, out_channels, height, width) or (2, 8, 23, 23), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((23 + 2*0 - 3) / 2 + 1) = floor(11) = 11 for shape of (batch, out_channels, height, width) or (2, 8, 11, 11)
    nn.Conv2d(8, 16, kernel_size=5, padding=2), nn.ReLU(),           # output_size = floor((11 + 2*2 - 5) / 1 + 1) = floor(11) = 11 for shape of (batch, out_channels, height, width) or (2, 16, 11, 11), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((11 + 2*0 - 3) / 2 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 16, 5, 5)
    nn.Conv2d(16, 24, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 24, 5, 5), then ReLU
    nn.Conv2d(24, 24, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 24, 5, 5), then ReLU
    nn.Conv2d(24, 16, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 16, 5, 5), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((5 + 2*0 - 3) / 2 + 1) = floor(2) = 2 for shape of (batch, out_channels, height, width) or (2, 16, 2, 2)
    nn.Flatten(),                                                    # (2, 16, 2, 2) becomes (2, 16*2*2) or (2, 64)
    nn.Linear(16 * 2 * 2, 32), nn.ReLU(), nn.Dropout(p=0.5),         # (2, 64) @ (64, 32) = (2, 32), during training, ~50% of activations are randomly set to 0
    nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.5),                     # (2, 32) @ (32, 32) = (2, 32), during training, ~50% of activations are randomly set to 0
    nn.Linear(32, 10),                                               # (2, 32) @ (32, 10) = (2, 10)
)

* Chapter 7 ended with LeNet: a compact CNN that turns image grids into class logits.
* Chapter 8 starts the modern CNN tour.
* AlexNet matters because it showed that a **deeper CNN trained at large scale could learn useful visual representations directly from data instead of relying on hand-engineered features**.

# How to use this notebook

* Run the notebook from top to bottom in a clean kernel.

* The code uses small synthetic tensors so that architecture mechanics can be inspected without downloads, `torchvision`, ImageNet-scale images, or long training runs.

* Before important cells, predict the shape, parameter count, or failure mode, then read the assertions as executable contracts.

# You are done when you can

- explain representation learning in the AlexNet story
- trace how an AlexNet-style stack reduces spatial resolution and expands channels
- explain why ReLU helped deeper networks compared with saturating activations
- explain why dropout appears in the dense classifier head
- debug a fixed-flatten-size failure when the input resolution changes

In [3]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_parameters(module):
    return sum(p.numel() for p in module.parameters())

def trace_module_shapes(module, X):
    rows = []
    current = X

    for name, layer in module.named_children():
        current = layer(current)
        rows.append((name, layer.__class__.__name__, shape(current)))
    return rows, current

def conv2d_hw(height, width, kernel_size, stride=1, padding=0):
    h = math.floor((height + 2 * padding - kernel_size) / stride) + 1
    w = math.floor((width + 2 * padding - kernel_size) / stride) + 1
    return h, w

def poo2d_hw(height, width, kernel_size, stride):
    return conv2d_hw(height, width, kernel_size, stride, padding=0)

# 8.1.0 The Problem This Notebook Solves

LeNet already used convolution, nonlinear activation, pooling, flattening, and dense classification.

AlexNet scales that idea in a historically important way:

- larger early receptive fields
- more channels
- more convolutional layers
- ReLU activations instead of saturating sigmoids
- dropout in the dense classifier head
- large-scale supervised training

Representation learning means the model learns intermediate features from data.
* Earlier computer vision often **depended heavily on features written by humans, such as edge, texture, or shape descriptors**.
* AlexNet-style CNNs **learn many of those useful intermediate detectors as trainable parameters**.

This notebook does not reproduce AlexNet's ImageNet training (data, hardware, and time outside this chapter's purpose).

The goal here is to make the architecture mechanically readable.

# 8.1.1 Spatial Size Shrinks While Channels Grow

Modern CNNs often follow a repeated pattern:

```text
spatial resolution goes down
channel count goes up
semantic richness goes up
```
* Spatial resolution means height and width.
* Channel count means the number of feature maps at each spatial location.
* Early layers preserve more local detail; later layers store more abstract feature evidence across fewer locations.

Before running the cell, predict:

- The first large-stride convolution should reduce 96 by 96 sharply.
- Pooling should reduce spatial size again.
- Later convolutions with padding should preserve spatial size inside the stack.

In [ ]:
height, width = 96, 96
steps = [
    ("conv11 stride4 pad2", lambda h, w: conv2d_hw(h, w, 11, stride=4, padding=2)),    # floor((96 + 2*2 - 11) / 4 + 1) = floor(23.25) = 23; lambda takes h and w as inputs and passes them into conv2d_hw()
    ("pool3 stride2", lambda h, w: conv2d_hw(h, w, 3, stride=2)),                      # floor((23 + 2*0 - 3) / 2 + 1) = floor(11.5) = 11
    ("conv5 pad2", lambda h, w: conv2d_hw(h, w, 5, padding=2)),                        # floor((11 + 2*2 - 5) / 1 + 1) = floor(11) = 11
    ("pool3 stride2", lambda h, w: conv2d_hw(h, w, 3, stride=2)),                      # floor((11 + 2*0 - 3) / 2 + 1) = floor(5) = 5
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),                        # floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),                        # floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5
    ("conv3 pad1", lambda h, w: conv2d_hw(h, w, 3, padding=1)),                        # floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5
    ("pool3 stride2", lambda h, w: conv2d_hw(h, w, 3, stride=2)),                      # floor((5 + 2*0 - 3) / 2 + 1) = floor(2) = 2
]

trace = []
for name, update in steps:
    height, width = update(height, width) # For each item in steps, put the name into name and put the function into update
    trace.append((name, height, width))
print(trace)
assert trace[0][1:] == (23, 23) # The first resulting shape of the layer
assert trace[-1][1:] == (2, 2) # The final resulting shape of the layer

[('conv11 stride4 pad2', 23, 23), ('pool3 stride2', 11, 11), ('conv5 pad2', 11, 11), ('pool3 stride2', 5, 5), ('conv3 pad1', 5, 5), ('conv3 pad1', 5, 5), ('conv3 pad1', 5, 5), ('pool3 stride2', 2, 2)]


# 8.1.2 A Small AlexNet-Style Network

The real AlexNet used much larger channel counts and dense layers.

This version keeps the same architectural pattern while reducing width so that the forward pass is quick:

```text
large early convolution -> pooling -> deeper conv stack -> pooling -> dense head
```
* The dense head has a fixed input feature contract.
* For 96 by 96 inputs, the convolutional feature extractor below produces `(batch, 16, 2, 2)`, so flattening gives 64 features per example.

Before running the cell, predict:

- The output logits should have shape `(2, 10)`.
- The flattened size before the first linear layer should be 64.
- The model has trainable parameters in both the convolutional feature extractor and dense head.

In [ ]:
# conv2d(24, 24...) and linear(32, 32) are not redundant in these cases because they allowed the model more opportunities to learn sophisticated features before reducing the outputs (to 16 and 10)

alexnet_small = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=11, stride=4, padding=2), nn.ReLU(), # output_size = floor((96 + 2*2 - 11) / 4 + 1) = floor(23.25) = 23 for shape of (batch, out_channels, height, width) or (2, 8, 23, 23), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((23 + 2*0 - 3) / 2 + 1) = floor(11) = 11 for shape of (batch, out_channels, height, width) or (2, 8, 11, 11)
    nn.Conv2d(8, 16, kernel_size=5, padding=2), nn.ReLU(),           # output_size = floor((11 + 2*2 - 5) / 1 + 1) = floor(11) = 11 for shape of (batch, out_channels, height, width) or (2, 16, 11, 11), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((11 + 2*0 - 3) / 2 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 16, 5, 5)
    nn.Conv2d(16, 24, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 24, 5, 5), then ReLU
    nn.Conv2d(24, 24, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 24, 5, 5), then ReLU
    nn.Conv2d(24, 16, kernel_size=3, padding=1), nn.ReLU(),          # output_size = floor((5 + 2*1 - 3) / 1 + 1) = floor(5) = 5 for shape of (batch, out_channels, height, width) or (2, 16, 5, 5), then ReLU
    nn.MaxPool2d(kernel_size=3, stride=2),                           # output_size = floor((5 + 2*0 - 3) / 2 + 1) = floor(2) = 2 for shape of (batch, out_channels, height, width) or (2, 16, 2, 2)
    nn.Flatten(),                                                    # (2, 16, 2, 2) becomes (2, 16*2*2) or (2, 64)
    nn.Linear(16 * 2 * 2, 32), nn.ReLU(), nn.Dropout(p=0.5),         # (2, 64) @ (64, 32) = (2, 32), during training, ~50% of activations are randomly set to 0
    nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.5),                     # (2, 32) @ (32, 32) = (2, 32), during training, ~50% of activations are randomly set to 0
    nn.Linear(32, 10),                                               # (2, 32) @ (32, 10) = (2, 10)
)

X = torch.randn(2, 1, 96, 96)
rows, logits = trace_module_shapes(alexnet_small, X)
for row in rows:
    print(row)

assert shape(logits) == (2, 10)
assert count_parameters(alexnet_small) > 0

('0', 'Conv2d', (2, 8, 23, 23))
('1', 'ReLU', (2, 8, 23, 23))
('2', 'MaxPool2d', (2, 8, 11, 11))
('3', 'Conv2d', (2, 16, 11, 11))
('4', 'ReLU', (2, 16, 11, 11))
('5', 'MaxPool2d', (2, 16, 5, 5))
('6', 'Conv2d', (2, 24, 5, 5))
('7', 'ReLU', (2, 24, 5, 5))
('8', 'Conv2d', (2, 24, 5, 5))
('9', 'ReLU', (2, 24, 5, 5))
('10', 'Conv2d', (2, 16, 5, 5))
('11', 'ReLU', (2, 16, 5, 5))
('12', 'MaxPool2d', (2, 16, 2, 2))
('13', 'Flatten', (2, 64))
('14', 'Linear', (2, 32))
('15', 'ReLU', (2, 32))
('16', 'Dropout', (2, 32))
('17', 'Linear', (2, 32))
('18', 'ReLU', (2, 32))
('19', 'Dropout', (2, 32))
('20', 'Linear', (2, 10))


| Tutorial `alexnet_small` | Original AlexNet           |
| ------------------------ | -------------------------- |
| 1 input channel          | 3 RGB channels             |
| 5 convolutional layers   | 5 convolutional layers     |
| ReLU activations         | ReLU activations           |
| 3 max-pooling layers     | 3 max-pooling layers       |
| 3 linear layers          | 3 fully connected layers   |
| Dropout in classifier    | Dropout in classifier      |
| 10 outputs               | 1,000 outputs              |
| Tiny channel counts      | 96 → 256 → 384 → 384 → 256 |
| Tiny FC layers (32)      | 4,096 → 4,096              |
| ~small parameter count   | ~60M parameters            |


# 8.1.3 ReLU Keeps a Stronger Gradient in the Positive Region

A saturating activation is an activation whose derivative becomes tiny over a wide input range.

Sigmoid is useful in some places, but deep stacks of sigmoids can make gradient flow weak when activations saturate near 0 or 1.

The core reason ReLU performs better than Sigmoid in deep networks is that it helps **avoid the vanishing-gradient problem**.
* Sigmoid squashes activations between 0 and 1, and its derivatives are always ≤ 0.25, so as gradients are backpropagated through many layers, they can become progressively smaller.
* ReLU instead sets negative activations to zero while preserving positive activations unchanged, giving a gradient of 1 for positive inputs and allowing gradients to pass through without being diminished.

ReLU is simple:

```text
relu(x) = max(0, x)
```

* For positive inputs, its derivative is 1.
* That does not solve every optimization problem, but it makes deep networks easier to train than if every layer repeatedly squeezed values into a saturated range.

The cell compares gradients through a tiny activation-only computation.

This is not a full training proof. It isolates one mechanical difference.

In [ ]:
values = torch.tensor([-6.0, -1.0, 0.0, 1.0, 6.0], requires_grad=True)

sigmoid_loss = torch.sigmoid(values).sum()
sigmoid_loss.backward()
sigmoid_grads = values.grad.clone()

values.grad.zero_()
relu_loss = F.relu(values).sum()
relu_loss.backward()
relu_grads = values.grad.clone()

print("sigmoid grads:", sigmoid_grads)
print("relu grads:", relu_grads)

assert relu_grads[-1].item() == 1.0
assert sigmoid_grads[-1].item() < 0.01

sigmoid grads: tensor([0.0025, 0.1966, 0.2500, 0.1966, 0.0025])
relu grads: tensor([0., 0., 0., 1., 1.])


# 8.1.4 Dropout Changes Training Behavior, Not Evaluation Behavior

Dropout randomly zeros some activations during training.

The purpose is regularization: it **makes the dense classifier head less able to rely on one brittle co-adaptation of hidden units**.

Two practical rules matter:

- `model.train()` enables dropout randomness.
- `model.eval()` **disables dropout randomness and uses the full representation**.

This is a software-state issue, not just a mathematical layer.

A model that accidentally stays in training mode during inference can produce unstable predictions.

In [ ]:
drop = nn.Dropout(p=0.5)
X = torch.ones(12)

torch.manual_seed(1)
drop.train() # During training, each element has a 50% probability of being set to 0
Y_train = drop(X)

drop.eval() # All elements are preserved in their original values
Y_eval = drop(X)

print("training output:", Y_train)
print("eval output:", Y_eval)

assert (Y_train == 0).any() # Make sure that at least 1 element in Y_train is 0
assert torch.equal(Y_eval, X) # Y_eval should equal X because train()/eval() only switch dropout's behavior; dropout is disabled during eval()

# 8.1.5 Break It Deliberately: Fixed Flatten Size

The dense classifier sees a vector, not an image.
* If the convolutional stack produces a different spatial size, the flattened vector length changes.
* A fixed `Linear(in_features, out_features)` layer will then reject the input.

This is one of the most common CNN architecture mistakes:

```text
changed input resolution
changed convolution/pooling output size
forgot to update first dense layer
```

The cell intentionally feeds the small AlexNet a different input resolution and catches the failure so the notebook can continue.

In [ ]:
bad_input = torch.randn(2, 1, 80, 80)

try:
    alexnet_small(bad_input) # The model was designed around a 96×96 input, which produces 64 flattened features (16 × 2 × 2)
                             # Reducing the input from 96×96 to 80×80 causes the convolution/pooling stack to produce only 16 flattened features (16 × 1 × 1)
                             # but the Linear(64, 32) layer is still expecting the 64 features produced by the original 96×96 input
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("Expected the dense layer to reject the changed flatten size")

RuntimeError
mat1 and mat2 shapes cannot be multiplied (2x16 and 64x32)


# 8.1 Checkpoint

Answer these before moving on.

Short markdown answers in the notebook are enough; the checkpoint is meant to test whether you can explain the mechanics without rereading the code.

1. What does representation learning mean in the AlexNet story?
> * Representation learning means the network automatically learns increasingly useful features from the raw pixels through its layers, rather than having those features manually designed
> * AlexNet demonstrated the power of **learning rich representations from data using a large number of learnable parameters, enabled by more data and computation**

2. Why do modern CNNs often reduce spatial size while increasing channel count?
> CNNs reduce spatial resolution to **reduce computation and progressively capture larger-scale patterns**, while increasing channels gives the network more capacity to **represent different learned features**

3. Why did ReLU help deeper CNNs compared with saturating activations?
> * Sigmoid can cause vanishing gradients because its derivative becomes very small in saturated regions, and these small gradients compound across layers
> * ReLU has a derivative of 1 for positive inputs, allowing gradients to propagate more effectively

4. What does dropout do differently in training and evaluation modes?
> Dropout randomly drops ~drop rate % of elements inside a tensor to 0 during training, but keeps the fidelity of all elements during eval. This is meant to reduce single param dependency during training

5. Why can changing image resolution break a fixed dense classifier head?
> A fixed dense layer expects a fixed number of flattened features. Changing the image resolution changes the spatial dimensions produced by the convolution/pooling layers, which can change the number of features produced by `Flatten()`, causing a mismatch with the dense layer's expected `in_features`